In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 1.7: Stacked Autoencoder (Deep Feature Hierarchy)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build a deep stacked AE (5 encoder layers, 5 decoder layers)
#   - Understand WHY depth enables hierarchical feature learning
#   - Observe that depth alone is not magic — skip connections help
#   - Apply to image search/retrieval using learned latent features
#   - Build a nearest-neighbour retrieval system from Fashion-MNIST
#
# PREREQUISITES: 06_convolutional_ae.py
# ESTIMATED TIME: ~20 min
#
# TASKS:
#   1. Build Stacked AE: 784 -> 512 -> 256 -> 128 -> 64 -> 16
#   2. Train and compare to shallower undercomplete AE
#   3. Visualise feature hierarchy via layer activations
#   4. Apply: image retrieval using latent space nearest neighbours
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F



## THEORY — Depth Enables Feature Hierarchy


In [ ]:
# Stack multiple autoencoder layers to learn a hierarchy of features:
# low-level (edges) -> mid-level (textures) -> high-level (shapes).
# Each layer learns to encode the previous layer's output.
#
# Analogy: In document processing, features exist at multiple levels.
# Character-level features detect handwriting quality; word-level
# features detect names and addresses; document-level features detect
# document type (passport vs utility bill). A stacked AE learns this
# hierarchy automatically.
#
# CAVEAT: Depth without skip connections can cause vanishing gradients.
# The value of stacking is in FEATURE HIERARCHY — the intermediate
# representations at each layer capture increasingly abstract features.



## TASK 1 — Load data and engines


In [ ]:
X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader = load_fashion_mnist()
conn, tracker, exp_name, registry, has_registry = setup_engines()



## TASK 2 — Build and Train Stacked AE


Deep encoder with 5 layers of progressive compression.


In [ ]:
class StackedAE(nn.Module):

    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        # TODO: Build encoder — nn.Sequential with 5 linear layers:
        #       784 -> 512 -> 256 -> 128 -> 64 -> latent_dim
        #       Use ReLU between each layer (not after last)
        self.encoder = ____

        # TODO: Build decoder — mirror of encoder:
        #       latent_dim -> 64 -> 128 -> 256 -> 512 -> input_dim
        #       ReLU between layers, Sigmoid at end
        self.decoder = ____

    def forward(self, x):
        # TODO: Encode then decode. Return (reconstruction, latent_code)
        ____


def stacked_ae_loss(model, xb):
    # TODO: Forward, MSE loss. Return (loss, {})
    ____


print("\n" + "=" * 70)
print("  Stacked AE — Deep Feature Hierarchy")
print("=" * 70)
print("  5 encoder layers: 784->512->256->128->64->16")

# TODO: Create StackedAE(INPUT_DIM, LATENT_DIM) and train
stacked_model = ____
stacked_losses = ____



## TASK 3 — Visualise


In [ ]:
# TODO: show_reconstruction
____



### Checkpoint


In [ ]:
assert len(stacked_losses) == EPOCHS
assert stacked_losses[-1] < stacked_losses[0]
# INTERPRETATION: More layers does NOT automatically mean better
# reconstruction. Depth without skip connections can cause vanishing
# gradients. The value of stacking is in FEATURE HIERARCHY.
print("\n--- Checkpoint passed --- stacked AE trained\n")

if has_registry:
    register_model(registry, "stacked_ae", stacked_model, stacked_losses[-1])



## APPLY — Image Search / Feature Extraction


In [ ]:
# BUSINESS SCENARIO: You are building a visual search engine for a
# Singapore fashion marketplace. Users upload a photo of clothing
# and want to find "similar items" from the catalogue. The stacked
# AE's latent space provides a 16-dimensional feature vector per
# image — use nearest-neighbour search to find similar items.
#
# The depth of the stacked AE means the 16-dim vector captures
# HIERARCHICAL features: Layer 1 encodes edges, Layer 3 encodes
# shapes, Layer 5 encodes category-level semantics. This multi-level
# encoding makes the similarity search more meaningful than pixel
# comparison or shallow features.

print("\n" + "=" * 70)
print("  APPLICATION: Image Search via Latent Space Retrieval")
print("=" * 70)

# --- Encode all test images to latent space ---
stacked_model.eval()
with torch.no_grad():
    test_latents = stacked_model.encoder(X_test_flat.to(device)).cpu().numpy()

print(f"Encoded {len(test_latents)} images to {test_latents.shape[1]}-dim latent space")

# --- Get labels for evaluation ---
_, test_labels = get_fashion_mnist_labels()
test_labels_np = test_labels.numpy()

CLASS_NAMES = [
    "T-shirt",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle Boot",
]

# --- Nearest-neighbour retrieval ---
from scipy.spatial.distance import cdist

query_indices = [0, 100, 200, 500, 1000, 2000, 3000, 5000, 7000, 9000]
n_retrieve = 8

# TODO: For each query, compute distances to all test images,
# find top-8 nearest neighbours, compute same-class precision.
# Create retrieval grid: query image + 8 retrieved images per row.
# Green border = same class, red border = different class.
# Save to OUTPUT_DIR / "ex1_image_retrieval.png"
fig, axes = plt.subplots(
    len(query_indices), n_retrieve + 1, figsize=(18, 2 * len(query_indices))
)

retrieval_precisions = []
for row, qi in enumerate(query_indices):
    # TODO: Compute Euclidean distances from query to all test latents
    # Exclude self (set dist[qi] = inf)
    # Find nearest n_retrieve indices
    # Compute precision (fraction of retrieved with same label as query)
    ____

mean_precision = np.mean(retrieval_precisions)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ex1_image_retrieval.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nRetrieval precision@{n_retrieve}: {mean_precision:.1%}")

# --- Latent space clustering visualisation (PCA to 2D) ---
from numpy.linalg import svd

# TODO: PCA projection of 3000 sample latents to 2D
# Scatter plot coloured by class label
# Save to OUTPUT_DIR / "ex1_stacked_ae_latent_clusters.png"
sample_size = 3000
sample_idx = np.random.choice(len(test_latents), sample_size, replace=False)
sample_latents = test_latents[sample_idx]
sample_labels = test_labels_np[sample_idx]

centered = sample_latents - sample_latents.mean(axis=0)
_, _, Vt = svd(centered, full_matrices=False)
projected = centered @ Vt[:2].T

fig, ax = plt.subplots(figsize=(10, 8))
____
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_stacked_ae_latent_clusters.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Business Impact ---
print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — Fashion Image Search Engine")
print("=" * 64)
print(f"\nRetrieval precision@{n_retrieve}: {mean_precision:.1%}")
print(
    f"Latent dimension: {LATENT_DIM} (vs 784 raw pixels = {784/LATENT_DIM:.0f}x smaller)"
)
print(f"\nFor a catalogue of 1M products:")
print(f"  Raw pixel search:     784 dims x 1M = 3.0 GB index")
print(
    f"  Latent space search:  {LATENT_DIM} dims x 1M = {LATENT_DIM * 4 / 1e6:.1f} MB index"
)
print(f"  Index size reduction: {784 / LATENT_DIM:.0f}x smaller")
print(
    f"  Search speed:         ~{784 / LATENT_DIM:.0f}x faster (fewer distance computations)"
)
print("=" * 64)



## REFLECTION


[x] Built a 5-layer stacked autoencoder for deep feature hierarchy
  [x] Understood that depth enables multi-level abstraction
  [x] Applied latent features to image search/retrieval
  [x] Built nearest-neighbour retrieval with precision evaluation
  [x] Visualised latent space clustering (PCA 2D projection)
  [x] Quantified index size and search speed improvements

  KEY INSIGHT: Depth enables HIERARCHY. Layer 1 learns edges, Layer 3
  learns shapes, Layer 5 learns category semantics. The 16-dim latent
  vector is a compact summary that preserves this hierarchy. For image
  search, this means retrieving items that are semantically similar
  (same type of clothing) rather than just visually similar (same
  brightness distribution).

  Next: 08_recurrent_ae.py handles sequential/time-series data...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments (depth stress-test)


In [ ]:
# 5-layer encoder = the DEEPEST dense AE in this exercise. Vanishing
# gradients are the primary risk at this depth without residuals.
# This is where the Blood Test earns its keep.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    xb = batch[0] if isinstance(batch, (tuple, list)) else batch
    loss, _ = stacked_ae_loss(m, xb)
    return loss


print("\n── Diagnostic Report (Stacked AE) ──")
diag, findings = run_diagnostic_checkpoint(
    stacked_model,
    flat_loader,
    _diag_loss,
    title="Stacked AE (5-layer)",
    n_batches=8,
    train_losses=stacked_losses,
    show=False,
)

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
#   [X] Gradient flow (CRITICAL): Vanishing gradients at
#       'encoder.3.weight' — min RMS = 4.2e-06, min
#       update_ratio = 8.9e-05. (Shallow layer encoder.0 RMS
#       ~3.1e-03 — ~750x spread across 5 layers.)
#       Fix: Kaiming init, GELU, or add skip/residual
#            connections between encoder blocks.
#   [!] Dead neurons  (WARNING): 'encoder.3' (relu): 44% dead.
#       Bottleneck ReLU saturation compounds vanishing
#       gradients — the two failure modes feed each other.
#   [✓] Loss trend    (HEALTHY): Loss converging
#       (train slope -1.1e-03/epoch) — slower than shallow
#       baseline (02) because deep layers are barely updating.



## Final train loss: ~0.018 after 10 epochs, 5-layer encoder.

STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:

 [BLOOD TEST — VANISHING GRADIENT TEXTBOOK CASE] 750x spread
    in RMS across 5 layers (encoder.0: 3.1e-03 → encoder.3:
    4.2e-06) is the canonical deep-dense-network failure.
    Slide 5K walks through the math: each ReLU + Linear
    layer multiplies the gradient by an expectation < 1,
    and 5 layers gets you ~0.5^5 = ~3% of the shallow-layer
    magnitude. Depth without skip connections is how AlexNet
    barely worked and ResNet solved it.
    >> Prescription: Three fixes of increasing impact:
       (a) Kaiming He init (partial — slows the collapse)
       (b) GELU or SiLU (fully non-saturating on negatives)
       (c) Residual connections (FIXES it — gradient
           bypasses depth via additive skip). Applied in
           ex_2/02_resnet_se.py — forward reference.

 [X-RAY] 44% dead at encoder.3 is the compounding failure:
    if gradient is vanishing, dead ReLUs never recover
    (the tiny gradient cannot re-activate them), and the
    remaining live ReLUs overload to compensate, pushing
    more of them into saturation next batch. Positive
    feedback loop into deeper dysfunction.
    >> Prescription: LeakyReLU (negative-slope leak lets
       small gradients revive dead channels) OR a residual
       block (additive skip bypasses the ReLU gate).

 [STETHOSCOPE] Loss still DECREASES despite dysfunction —
    this is the trap. Final loss 0.018 looks "fine" but
    compare to 06_convolutional (~0.0048) at similar latent
    size. The loss curve lies when the architecture is
    pathological; only the Blood Test + X-Ray reveal that
    most of the model isn't learning.
    >> Prescription: ALWAYS read 3 instruments. The
       Stethoscope alone would have shipped this model.

 FIVE-INSTRUMENT TAKEAWAY: stacked AE is the "how NOT to go
 deep" cautionary tale. The fix isn't more training or more
 regularisation — it's ARCHITECTURAL (skip connections).
 This motivates ex_2 ResNet-SE and foreshadows the same
 pattern in ex_3 RNNs (where "skip" becomes LSTM/GRU gating).


## TASK 3 — Visualise


In [ ]:
show_reconstruction(stacked_model, X_test_flat, "Stacked AE (5 Layers)")



### Checkpoint


In [ ]:
assert len(stacked_losses) == EPOCHS
assert stacked_losses[-1] < stacked_losses[0]
# INTERPRETATION: More layers does NOT automatically mean better
# reconstruction. Depth without skip connections can cause vanishing
# gradients. The value of stacking is in FEATURE HIERARCHY.
print("\n--- Checkpoint passed --- stacked AE trained\n")

if has_registry:
    register_model(registry, "stacked_ae", stacked_model, stacked_losses[-1])



## APPLY — Image Search / Feature Extraction


In [ ]:
# BUSINESS SCENARIO: You are building a visual search engine for a
# Singapore fashion marketplace. Users upload a photo of clothing
# and want to find "similar items" from the catalogue. The stacked
# AE's latent space provides a 16-dimensional feature vector per
# image — use nearest-neighbour search to find similar items.
#
# The depth of the stacked AE means the 16-dim vector captures
# HIERARCHICAL features: Layer 1 encodes edges, Layer 3 encodes
# shapes, Layer 5 encodes category-level semantics. This multi-level
# encoding makes the similarity search more meaningful than pixel
# comparison or shallow features.

print("\n" + "=" * 70)
print("  APPLICATION: Image Search via Latent Space Retrieval")
print("=" * 70)

# --- Encode all test images to latent space ---
stacked_model.eval()
with torch.no_grad():
    test_latents = stacked_model.encoder(X_test_flat.to(device)).cpu().numpy()

print(f"Encoded {len(test_latents)} images to {test_latents.shape[1]}-dim latent space")

# --- Get labels for evaluation ---
_, test_labels = get_fashion_mnist_labels()
test_labels_np = test_labels.numpy()

CLASS_NAMES = [
    "T-shirt",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle Boot",
]

# --- Nearest-neighbour retrieval ---
from scipy.spatial.distance import cdist

# Sample 10 query images
query_indices = [0, 100, 200, 500, 1000, 2000, 3000, 5000, 7000, 9000]
n_retrieve = 8  # top-8 similar

fig, axes = plt.subplots(
    len(query_indices), n_retrieve + 1, figsize=(18, 2 * len(query_indices))
)
fig.suptitle(
    "Image Retrieval: Query (left) and Top-8 Nearest Neighbours (right)\n"
    "Green border = same class, Red border = different class",
    fontsize=13,
    y=1.02,
)

retrieval_precisions = []

for row, qi in enumerate(query_indices):
    query_latent = test_latents[qi : qi + 1]
    dists = cdist(query_latent, test_latents, metric="euclidean")[0]
    # Exclude self
    dists[qi] = float("inf")
    nearest_idx = np.argsort(dists)[:n_retrieve]

    query_label = test_labels_np[qi]
    retrieved_labels = test_labels_np[nearest_idx]
    precision = (retrieved_labels == query_label).mean()
    retrieval_precisions.append(precision)

    # Plot query
    axes[row, 0].imshow(X_test_flat[qi].cpu().reshape(28, 28), cmap="gray")
    axes[row, 0].set_title(f"Query\n{CLASS_NAMES[query_label]}", fontsize=8)
    axes[row, 0].axis("off")

    # Plot retrieved
    for col, ni in enumerate(nearest_idx):
        axes[row, col + 1].imshow(X_test_flat[ni].cpu().reshape(28, 28), cmap="gray")
        match = test_labels_np[ni] == query_label
        border_color = "#4CAF50" if match else "#F44336"
        for spine in axes[row, col + 1].spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(3)
        axes[row, col + 1].set_title(
            f"{CLASS_NAMES[test_labels_np[ni]]}\nd={dists[ni]:.2f}", fontsize=7
        )
        axes[row, col + 1].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ex1_image_retrieval.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR / 'ex1_image_retrieval.png'}")

mean_precision = np.mean(retrieval_precisions)
print(f"\nRetrieval precision@{n_retrieve}: {mean_precision:.1%}")

# --- Latent space clustering visualisation (PCA to 2D) ---
from numpy.linalg import svd

sample_size = 3000
sample_idx = np.random.choice(len(test_latents), sample_size, replace=False)
sample_latents = test_latents[sample_idx]
sample_labels = test_labels_np[sample_idx]

centered = sample_latents - sample_latents.mean(axis=0)
_, _, Vt = svd(centered, full_matrices=False)
projected = centered @ Vt[:2].T

fig, ax = plt.subplots(figsize=(10, 8))
for cls in range(10):
    mask = sample_labels == cls
    ax.scatter(
        projected[mask, 0], projected[mask, 1], s=10, alpha=0.5, label=CLASS_NAMES[cls]
    )
ax.set_xlabel("PC1", fontsize=12)
ax.set_ylabel("PC2", fontsize=12)
ax.set_title(
    "Stacked AE Latent Space (PCA 2D Projection)\nClusters show semantic grouping",
    fontsize=13,
)
ax.legend(fontsize=8, markerscale=3, ncol=2, loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_stacked_ae_latent_clusters.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Business Impact ---
print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — Fashion Image Search Engine")
print("=" * 64)
print(f"\nRetrieval precision@{n_retrieve}: {mean_precision:.1%}")
print(
    f"Latent dimension: {LATENT_DIM} (vs 784 raw pixels = {784/LATENT_DIM:.0f}x smaller)"
)
print(f"\nFor a catalogue of 1M products:")
print(f"  Raw pixel search:     784 dims x 1M = 3.0 GB index")
print(
    f"  Latent space search:  {LATENT_DIM} dims x 1M = {LATENT_DIM * 4 / 1e6:.1f} MB index"
)
print(f"  Index size reduction: {784 / LATENT_DIM:.0f}x smaller")
print(
    f"  Search speed:         ~{784 / LATENT_DIM:.0f}x faster (fewer distance computations)"
)
print(f"\nSearch quality:")
print(
    f"  Same-class retrieval: {mean_precision:.0%} of top-{n_retrieve} results match query class"
)
print(f"  The latent space groups semantically similar items together")
print(f"  (see PCA cluster plot: shoes cluster, shirts cluster, etc.)")
print(f"\nKey insight: The stacked AE's depth means its 16-dim features")
print(f"capture HIERARCHICAL semantics — not just pixel similarity but")
print(f"category-level similarity. A coat retrieves other coats, not")
print(f"just images with similar pixel distributions.")
print("=" * 64)



## REFLECTION


[x] Built a 5-layer stacked autoencoder for deep feature hierarchy
  [x] Understood that depth enables multi-level abstraction
  [x] Applied latent features to image search/retrieval
  [x] Built nearest-neighbour retrieval with precision evaluation
  [x] Visualised latent space clustering (PCA 2D projection)
  [x] Quantified index size and search speed improvements

  KEY INSIGHT: Depth enables HIERARCHY. Layer 1 learns edges, Layer 3
  learns shapes, Layer 5 learns category semantics. The 16-dim latent
  vector is a compact summary that preserves this hierarchy. For image
  search, this means retrieving items that are semantically similar
  (same type of clothing) rather than just visually similar (same
  brightness distribution).

  Next: 08_recurrent_ae.py handles sequential/time-series data...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

